In [ ]:
!pip install video2numpy


In [ ]:
import numpy as np
from google.colab import drive
import os
drive.mount('/content/drive')
%cd /content/drive/MyDrive/DeeptraceReward

KeyboardInterrupt: 

In [ ]:
!ffmpeg -version


In [ ]:
from video2numpy import video2numpy
frames = video2numpy('/content/drive/MyDrive/DeeptraceReward/real_mini/0_30_s_academic_v0_1_215-A4pTxrrBtMU-split_2.mp4')
print(type(frames))

In [ ]:
!ffprobe -v error -count_frames -select_streams v:0 -show_entries stream=nb_read_frames -of default=nokey=1:noprint_wrappers=1 "/content/drive/MyDrive/DeeptraceReward/real_mini/0_30_s_academic_v0_1_215-A4pTxrrBtMU-split_2.mp4"


In [ ]:
!pip install torch torchvision timm
!pip install pillow numpy matplotlib


In [ ]:
import torch
import torch.nn as nn
import timm
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from timm.utils import AttentionExtract
from torchvision import transforms
from timm.data import create_transform
from typing import Dict
from torch.nn import functional as F

timm.layers.set_fused_attn(False)

# Load the pre-trained DINOv2 model
model = timm.create_model('vit_small_patch14_dinov2.lvd142m', pretrained=True)
model.eval()

# Define the transformation
transform = transforms.Compose([
    transforms.Resize(518),
    transforms.CenterCrop(518),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load and preprocess the image
# Load image from URL

real_processed = '/content/drive/MyDrive/DeeptraceReward/real_mini_processed/'
fake_processed = '/content/drive/MyDrive/DeeptraceReward/fake_mini_processed/'

real_files = sorted([f for f in os.listdir(real_processed) if f.endswith('.npy')])
fake_files = sorted([f for f in os.listdir(fake_processed) if f.endswith('.npy')])

real_features = '/content/drive/MyDrive/DeeptraceReward/real_mini_features/'
fake_features = '/content/drive/MyDrive/DeeptraceReward/fake_mini_features/'


# helper function to apply feature extraction on each frame of the np array that represents a video
def extract_video_embedding(frames: np.ndarray) -> np.ndarray:
    """Compute mean DINOv2 embedding for all frames in a video."""
    all_embeddings = []
    with torch.no_grad(): # saves memory and runtime bc we don't need to keep track of gradient
        for frame in frames:
            # Convert to PIL and transform
            image = Image.fromarray(frame.astype(np.uint8)) # not sure if we lose info going from float in nparray to ints in PIL
            img_tensor = transform(image).unsqueeze(0)  # (1, 3, H, W)
            emb = model(img_tensor)                    # (1, 384)
            all_embeddings.append(emb.squeeze(0))      # (384,)
    # Average embeddings across frames
    stacked = torch.stack(all_embeddings, dim=0)       # (T, 384)
    video_embedding = stacked.mean(dim=0)              # (384,) is median supposedly better for temporal aspect
    return video_embedding.cpu().numpy()

for fname in fake_files:
    path = os.path.join(fake_processed, fname)
    frames = np.load(path)
    print(f"Processing FAKE video: {fname}, frames={frames.shape}")
    emb = extract_video_embedding(frames)
    np.save(os.path.join(fake_features, fname.replace('.npy', '_feat.npy')), emb)



